<a href="https://colab.research.google.com/github/BRNA2000/ApachOpenwhisk/blob/master/datasetcharging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import glob
import os
# 1. Define the path to our invocation folder
#path = 'D:/DOCS/CONTAINER COLD START LATENCY/ourSolution/azurefunctions-dataset2019/invocation' # Change this to your folder path
#path = '/home/azurefunctions-dataset2019/invocation' # files access path
path = '/home/invocation' # files access path

all_files = glob.glob(os.path.join(path, "*.csv")) # indicates all files in specified access path



#++++++++++++++++++++++++++++++++++++++++++++++++++
# 2. Create a list of DataFrames and add a 'source' column to track origin
df_list = []
i=0
for filename in all_files:
    df = pd.read_csv(filename, low_memory=False) # , usecols=cols_voulues)
    #CONVERTIR  Toutes les colonnes int64 → int32
    int_cols = df.select_dtypes(include="int64").columns
    df[int_cols] = df[int_cols].astype("int16")
    i += 1
    df_copy = df.copy()  # forcer une vraie copie
    df_copy["jour"] = i
    #df['jour'] =i+1 #os.path.basename(filename) # Helpful for debugging
    #5.  pass from wide format to long format
    id_cols = ["HashOwner", "HashApp", "HashFunction","Trigger", "jour"],  # colonnes à garder
    # Colonnes des minutes (1 à 1440)
    #minute_cols = [str(i) for i in range(1, 1441)]  # ['1', '2', ..., '1440']
    minute_cols = [c for c in df_copy.columns if c not in id_cols]

    # Passer en format long
    print("ramined cols 1.")
    df_long = df_copy.melt(
        id_vars=id_cols,
        value_vars=minute_cols,               # colonnes à transformer
        var_name="minute",                    # nom de la nouvelle colonne
        value_name="nb_invocations"           # nom des valeurs
    )
    print("ramined cols 2.")
    # Convertir minute en entier et trier
    df_long["minute"] = df_long["minute"].astype("int16")
    #df_long = df_long.sort_values(["HashOwner", "HashApp","HashFunction", "minute"]).reset_index(drop=True)

    print(df_long.shape)
    df_long.head(10)


#df_list.append(df_copy)
df_list.append(df_long)
print(i)
print(f"current df .")

# 3. Concatenate all DataFrames into one
master_df = pd.concat(df_list, axis=0, ignore_index=True)

# 4. Preview the result
print(f"Combined {len(all_files)} files.")
master_df.head()





#5. merge with function duration
# Charger le 2ème fichier
#df2 = pd.read_csv("ton_fichier.csv", low_memory=False)

# LEFT JOIN sur 2 colonnes communes
#master_df = pd.merge(
 #   master_df,      # fichier gauche (garde toutes ses lignes)
  #  df2,            # fichier droit
  #  on=["colonne1", "colonne2"],  # ✅ remplace par tes vrais noms
   # how="left"      # LEFT JOIN
#)

ramined cols 1.


TypeError: unhashable type: 'list'

In [2]:
# Exploratory
master_df.shape

(2877120, 6)

In [3]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2877120 entries, 0 to 2877119
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   HashOwner       object 
 1   HashApp         object 
 2   HashFunction    object 
 3   Trigger         object 
 4   minute          int16  
 5   nb_invocations  float64
dtypes: float64(1), int16(1), object(4)
memory usage: 115.2+ MB


In [4]:
# Exploratory
master_df.describe()

,minute,nb_invocations
count,2.877120e+06,2.876299e+06
mean,7.205000e+02,4.839968e+00
std,4.156922e+02,7.104693e+01
min,1.000000e+00,0.000000e+00
25%,3.607500e+02,0.000000e+00
50%,7.205000e+02,0.000000e+00
75%,1.080250e+03,0.000000e+00
max,1.440000e+03,2.741400e+04


In [5]:
nan_df = pd.DataFrame({
    "nb_nan"   : master_df.isnull().sum(),
    "pourcentage" : (master_df.isnull().mean() * 100).round(2)
})

# Afficher seulement les colonnes avec NaN
print(nan_df[nan_df["nb_nan"] > 0].sort_values("pourcentage", ascending=False))

                nb_nan  pourcentage
nb_invocations     821         0.03


In [6]:
master_df.isnull()


,HashOwner,HashApp,HashFunction,Trigger,minute,nb_invocations
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
...,...,...,...,...,...,...
2877115,False,False,False,False,False,False
2877116,False,False,False,False,False,False
2877117,False,False,False,False,False,False
2877118,False,False,False,False,False,False
